#### Import Libraries

In [1]:
import pandas as pd
import numpy as np
import os
from faker import Faker
from datetime import datetime
import json

#### Data Generation

In [2]:
# Inicializa o Faker para gerar dados fictícios
fake = Faker('pt_BR')

# --- Funções de I/O (Simulando S3 localmente) ---
# (Reutilizando as funções da aula anterior)

def setup_directories():
    """Cria as pastas locais para simular o S3."""
    os.makedirs('data/acme_corp/bronze/contracts/it_contracts', exist_ok=True)
    os.makedirs('data/acme_corp/silver/contracts/it_contracts', exist_ok=True)
    os.makedirs('data/acme_corp/gold/contracts/it_contracts', exist_ok=True)

    os.makedirs('data/acme_corp/bronze/contracts/it_contracts_amount_history', exist_ok=True)
    os.makedirs('data/acme_corp/silver/contracts/it_contracts_amount_history', exist_ok=True)
    os.makedirs('data/acme_corp/gold/contracts/it_contracts_amount_history', exist_ok=True)

def write_parquet_local(df, path):
    """Função simplificada para 'upload' (salvar localmente)."""
    print(f"Salvando dados em: {path}")
    df.to_parquet(path, index=False)

def read_parquet_local(path):
    """Função simplificada para 'download' (ler localmente)."""
    print(f"Lendo dados de: {path}")
    if not os.path.exists(path):
        raise FileNotFoundError(f"Arquivo não encontrado: {path}. Rode a simulação de dados primeiro.")
    return pd.read_parquet(path)

# Roda a configuração
setup_directories()

# --- Funções Mock (Simulando APIs e S3) ---

def mock_s3_manipulations(kind_of_manipulation, df, PREFIX_KEY, credentials, bucket_name, dest_key):
    """
    Simula o 's3_manipulations' original, mas usa nossas funções locais.
    """
    if kind_of_manipulation == "parquet_upload":
        write_parquet_local(df, PREFIX_KEY)
    
    elif kind_of_manipulation == "parquet_download":
        return read_parquet_local(PREFIX_KEY)
    
    elif kind_of_manipulation == "copy_file":
        if dest_key is None:
            raise ValueError("Você precisa passar o parâmetro dest_key para copiar o arquivo.")
        # Simula a cópia lendo e escrevendo
        df_copy = read_parquet_local(PREFIX_KEY)
        write_parquet_local(df_copy, dest_key)
        print(f"📂 Arquivo 'copiado' para {dest_key}")

def mock_sharepoint_token_generation(credentials):
    """Simula a geração de token. Retorna um 'Response' falso."""
    print("Gerando token do SharePoint (Simulado)...")
    
    class FakeResponse:
        def json(self):
            return {"access_token": "FAKE_TOKEN_12345"}
            
    return FakeResponse()

In [7]:
def generate_fake_sharepoint_data(num_contracts=100):
    """
    Gera um DataFrame com dados "sujos" e problemas como viriam 
    do json_normalize de uma lista do SharePoint.
    """
    data = []
    
    for _ in range(num_contracts):
        # Problema 2: Listade analist as (pode ser 1, 2, ou nenhum)
        num_analistas = np.random.choice([0, 1, 2])
        analistas = []
        for i in range(num_analistas):
            analistas.append({"Title": fake.name()})
        if num_analistas == 0:
            analistas = None # Simula um campo vazio
            
        data.append({
            'ID': fake.unique.random_int(min=1000, max=5000),
            'Title': f"Contrato {fake.company()} {fake.license_plate()}",
            'Status': np.random.choice(['Ativo', 'Encerrado', 'Em Renovação', None]),
            'Fabricante': np.random.choice(['Microsoft', 'Oracle', 'Salesforce', 'SAP', 'AWS', 'Google']),
            'Revenda': np.random.choice(['ACME Ltda', 'Vendas Corp', 'Software S.A.']),
            'Departamento': np.random.choice(['TI Infra', 'TI Business', 'TI Operações']),
            'InicioVigencia': fake.date_between(start_date='-5y', end_date='-1y'),
            'FimVigencia': fake.date_between(start_date='+1y', end_date='+3y'),
            'RenovacaoAutomatica': np.random.choice([True, False]),
            'ValorAtual': fake.random_int(min=5000, max=500000),
            
            # Problema 3: Colunas "largas" (Wide)
            'Valor2018': np.random.choice([fake.random_int(min=1000, max=10000), np.nan]),
            'Valor2019': np.random.choice([fake.random_int(min=1000, max=10000), np.nan]),
            'Valor2020': np.random.choice([fake.random_int(min=1000, max=10000), np.nan]),
            'Valor2021': np.random.choice([fake.random_int(min=1000, max=10000), np.nan]),
            'Valor2022': np.random.choice([fake.random_int(min=1000, max=10000), np.nan]),
            'Valor2023': np.random.choice([fake.random_int(min=1000, max=10000), np.nan]),
            'Valor2024': np.random.choice([fake.random_int(min=1000, max=10000), np.nan]),
            
            'Moeda': 'BRL',
            
            # Problema 5: Números com Nulos
            'ContaRazao': np.random.choice([12345.0, 67890.0, 98765.0, np.nan]), # Float para simular
            
            # Problema 4: HTML Sujo
            'AtaRDG': np.random.choice(['Ata 123/2023', 'Ata 456/2022<br>Ata 789/2023', None]),
            
            'Modified': fake.date_time_this_year(),
            'Created': fake.date_time_this_decade(),
            
            # Problema 1: JSON Aninhado (Nível 1)
            'UsuarioChave.Title': np.random.choice([fake.name(), None]),
            'GestorResponsavel.Title': fake.name(),
            'Author.Title': fake.name(),
            'Editor.Title': fake.name(),
            
            # Problema 2: JSON Aninhado (Lista de Objetos)
            'AnalistaContrato': analistas
        })
        
    df = pd.DataFrame(data)
    # Simula colunas que não foram pedidas no $select e que queremos remover
    df['OData__ModerationStatus'] = 0 
    return df


def mock_read_list_from_sharepoint(site_url, list_name, token, odata_query=""):
    """
    Simula o 'read_list_from_sharepoint'. 
    Em vez de chamar a API, ela chama nosso gerador de dados falsos.
    """
    print(f"Iniciando leitura da lista: {list_name} (Simulado)")
    print(f"Com OData query: {odata_query[:100]}...") # Mostra parte da query
    
    # Geramos os dados falsos
    df = generate_fake_sharepoint_data()
    
    print(f"Leitura finalizada. Total de {len(df)} itens encontrados (Simulado).")
    return df

#### Construção da Camada BRONZE

In [10]:
def bronze_layer_construction(credentials, bucket_name):

    # --- 1. Descaracterização ---
    SITE_URL = "https://acme365.sharepoint.com/teams/IT-Contracts"
    LIST_NAME = "Contratos TI [Corporativo]" # Nome da lista
    
    # --- 2. Simulação (Usando mocks) ---
    # Usamos nossas funções simuladas
    access_token = mock_sharepoint_token_generation(credentials)
    token = access_token.json()["access_token"]

    # --- Lógica Original (Mantida) ---
    # A lógica de quais colunas buscar é a mesma
    contract_columns = [
        'ID', 'Title', 'Status', 'Fabricante', 'Revenda', 'Departamento', 'InicioContratacao', 'InicioVigencia',
        'FimVigencia', 'ProximoReajuste', 'RenovacaoAutomatica', 'ValorAtual',
        'Valor2030', 'Valor2029', 'Valor2028', 'Valor2027', 'Valor2026', 'Valor2025', 'Valor2024',
        'Valor2023', 'Valor2022', 'Valor2021', 'Valor2020', 'Valor2019', 'Valor2018', 'Valor2017', 'Valor2016',
        'Indexador', 'Moeda', 'Classe', 'ContaRazao', 'Abrangencia', 'AtaRDG', 'Alcada', 'PIS', 'Comentarios',
        'CartaoCredito', 'ProcessoIniciado', 'Modified', 'Created'
    ]
    users_columns = [
        'UsuarioChave',
        'AnalistaContrato',
        'GestorResponsavel',
        'Author',
        'Editor'
    ]
    colunas_select = contract_columns + [f"{user}/Title" for user in users_columns]
    colunas_expand = users_columns
    query_string = (
        f"$select={','.join(colunas_select)}"
        f"&$expand={','.join(colunas_expand)}"
    )

    # --- 3. Simulação (Usando mocks) ---
    contracts_df = mock_read_list_from_sharepoint(
        site_url=SITE_URL,
        list_name=LIST_NAME,
        token=token,
        odata_query=query_string
    )
    
    print("\n--- Amostra de Dados Brutos (Simulados) ---")
    print(contracts_df[['ID', 'AnalistaContrato', 'UsuarioChave.Title', 'AtaRDG', 'ContaRazao']].head())


#### Construção da Camada SILVER

In [ ]:
def silver_layer_construction(credentials, bucket_name):

    snapshot_date = datetime.now().strftime("%Y-%m-%d")

    # --- 1. Leitura (Descaracterizada) ---
    FILE_KEY = f'data/acme_corp/bronze/contracts/it_contracts/{snapshot_date}.parquet'
    try:
        contracts_df = mock_s3_manipulations("parquet_download", None, FILE_KEY, credentials, bucket_name, None)
    except FileNotFoundError:
        print(f"Erro! Arquivo Bronze não encontrado em {FILE_KEY}")
        print("Certifique-se de executar a Célula 3 (Camada Bronze) primeiro.")
        return

    print(f"Dados Bronze carregados. {len(contracts_df)} contratos.")
    
    # --- LÓGICA DE TRANSFORMAÇÃO (SILVER) ---

    # Bloco 1: Identificação Dinâmica de Colunas
    # Esta é uma ótima prática: o código se adapta se 'Valor2025', 'Valor2026' etc. aparecerem.
    year_columns = [
        col for col in contracts_df.columns 
        if col.startswith('Valor') and len(col) == 9 and col[5:].isdigit()
    ]
    print(f"Colunas de valor anual identificadas: {year_columns}")
    
    # Identifica as colunas de "features" (metadados do contrato)
    columns_without_history = [
        col for col in contracts_df.columns 
        if col not in year_columns
    ]

    # Bloco 2: Criação do DataFrame 1 (Features do Contrato)
    print("Processando DataFrame 1: Features do Contrato...")
    it_contracts_df = contracts_df[columns_without_history].copy()
    
    # Limpeza de HTML
    it_contracts_df['AtaRDG'] = it_contracts_df['AtaRDG'].str.replace('<br>', ';', regex=False)
    
    # Limpeza de Nulos (ContaRazao)
    # 1. Converte para Int64 (que suporta Nulos 'pd.NA') 
    it_contracts_df['ContaRazao'] = it_contracts_df['ContaRazao'].astype('Int64')
    # 2. Converte para string (agora 'pd.NA' vira '<NA>')
    it_contracts_df['ContaRazao'] = it_contracts_df['ContaRazao'].astype(str)
    # 3. Substitui o <NA> por string vazia
    it_contracts_df['ContaRazao'] = it_contracts_df['ContaRazao'].replace('<NA>', '')

    
    # Bloco 3: Criação do DataFrame 2 (Histórico de Valor)
    print("Processando DataFrame 2: Histórico de Valor (Melt)...")
    
    # Selecionamos as chaves + colunas de valor
    id_vars = ['ID', 'Title', 'Departamento', 'execution_date']
    large_historical_df = contracts_df[id_vars + year_columns]

    # A mágica do "unpivot"
    it_contracts_amount_history = large_historical_df.melt(
        id_vars=id_vars,
        var_name='year',     # Nova coluna para "Valor2023", "Valor2024"
        value_name='value'   # Nova coluna para os valores (10000, 12000)
    )

    # Limpeza pós-melt
    # 'Valor2024' -> 2024
    it_contracts_amount_history['year'] = it_contracts_amount_history['year'].str.replace('Valor', '').astype(int)
    # Remove linhas onde o valor era 'NaN' (contrato não tinha valor para aquele ano)
    it_contracts_amount_history = it_contracts_amount_history.dropna(subset=['value'])

    # --- 3. Salvamento (Descaracterizado) ---
    print("\nSalvando DataFrames Silver...")
    FILE_KEY_FEATURES = f'data/acme_corp/silver/contracts/{snapshot_date}.parquet'
    mock_s3_manipulations("parquet_upload", it_contracts_df, FILE_KEY_FEATURES, credentials, bucket_name, None)

    FILE_KEY_HISTORY = f'data/acme_corp/silver/contracts/it_contracts_amount_history/{snapshot_date}.parquet'
    mock_s3_manipulations("parquet_upload", it_contracts_amount_history, FILE_KEY_HISTORY, credentials, bucket_name, None)

    print("\n--- Processamento Silver Concluído ---")
    print("\nAmostra DataFrame 1 (Features):")
    print(it_contracts_df[['ID', 'Title', 'AtaRDG', 'ContaRazao']].head())
    
    print("\nAmostra DataFrame 2 (Histórico - Long):")
    print(it_contracts_amount_history.sample(5))

    return contracts_df, it_contracts_df, it_contracts_amount_history


# --- Executando a Camada Silver ---
contracts_df, it_contracts_df, it_contracts_amount_history = silver_layer_construction(credentials=None, bucket_name="acme-datalake")

In [18]:
contracts_df.head(2)

,ID,Title,Status,Fabricante,Revenda,Departamento,Usuario Chave,Analista Contrato,Gestor Responsavel,Criado Por,...,Valor2021,Valor2020,Valor2019,Valor2018,Moeda,ContaRazao,AtaRDG,Modified,Created,execution_date
0,2937,Contrato Rocha Araújo Ltda. XRL-3W17,None,Google,ACME Ltda,TI Business,None,Sra. Cecília Machado,Lavínia Monteiro,Breno Moura,...,3773.0,NaN,1920.0,1052.0,BRL,12345.0,Ata 456/2022<br>Ata 789/2023,2025-06-29 03:06:45,2020-05-19 22:03:14,2025-11-11
1,2950,Contrato Cirino XXU-7I18,None,AWS,ACME Ltda,TI Infra,None,None,Liz Melo,Cecilia Melo,...,5525.0,3732.0,NaN,2351.0,BRL,98765.0,Ata 456/2022<br>Ata 789/2023,2025-04-11 22:05:40,2022-03-19 20:27:38,2025-11-11


In [20]:
it_contracts_amount_history[it_contracts_amount_history['Title'] == 'Contrato Rocha Araújo Ltda. XRL-3W17'].head(50)

,ID,Title,Departamento,execution_date,year,value
0,2937,Contrato Rocha Araújo Ltda. XRL-3W17,TI Business,2025-11-11,2024,9594.0
100,2937,Contrato Rocha Araújo Ltda. XRL-3W17,TI Business,2025-11-11,2023,5031.0
200,2937,Contrato Rocha Araújo Ltda. XRL-3W17,TI Business,2025-11-11,2022,4185.0
300,2937,Contrato Rocha Araújo Ltda. XRL-3W17,TI Business,2025-11-11,2021,3773.0
500,2937,Contrato Rocha Araújo Ltda. XRL-3W17,TI Business,2025-11-11,2019,1920.0
600,2937,Contrato Rocha Araújo Ltda. XRL-3W17,TI Business,2025-11-11,2018,1052.0


In [ ]:
# --- LÓGICA DE TRANSFORMAÇÃO (BRONZE) ---
# Esta é a parte importante da aula: tratar o JSON aninhado

def extrair_nomes_de_lista(lista_de_usuarios):
    """
    Esta função "desaninha" uma coluna que contém uma LISTA de usuários
    e concatena os nomes.
    """
    if isinstance(lista_de_usuarios, list):
        nomes = []
        for usuario in lista_de_usuarios:
            if isinstance(usuario, dict) and 'Title' in usuario:
                nomes.append(usuario['Title'])
        return "; ".join(nomes) # Junta os nomes com "; "
    return pd.NA # Retorna 'Not Available' se não for uma lista

if 'AnalistaContrato' in contracts_df.columns:
    print("\nProcessando 'AnalistaContrato' (lista aninhada)...")
    contracts_df['Analista Contrato'] = contracts_df['AnalistaContrato'].apply(extrair_nomes_de_lista)
    contracts_df.drop(columns=['AnalistaContrato'], inplace=True)
else:
    print("Aviso: Coluna 'AnalistaContrato' não foi encontrada como esperado.")

# Renomeação de colunas "flat" (ex: UsuarioChave.Title -> Usuario Chave)
users_columns_to_rename = {
    'UsuarioChave.Title': 'Usuario Chave',
    'GestorResponsavel.Title': 'Gestor Responsavel',
    'Author.Title': 'Criado Por',
    'Editor.Title': 'Modificado Por'
}
contracts_df.rename(columns=users_columns_to_rename, inplace=True)

# Lógica de seleção de colunas final (mantida)
final_columns = [
    'ID', 'Title', 'Status', 'Fabricante', 'Revenda', 'Departamento', 'Usuario Chave', 'Analista Contrato',
    'Gestor Responsavel', 'Criado Por', 'Modificado Por', 'InicioContratacao', 'InicioVigencia',
    'FimVigencia', 'ProximoReajuste', 'RenovacaoAutomatica', 'ValorAtual',
    'Valor2030', 'Valor2029', 'Valor2028', 'Valor2027', 'Valor2026', 'Valor2025', 'Valor2024',
    'Valor2023', 'Valor2022', 'Valor2021', 'Valor2020', 'Valor2019', 'Valor2018', 'Valor2017', 'Valor2016',
    'Indexador', 'Moeda', 'Classe', 'ContaRazao', 'Abrangencia', 'AtaRDG', 'Alcada', 'PIS', 'Comentarios',
    'CartaoCredito', 'ProcessoIniciado', 'Modified', 'Created'
]
colunas_finais_existentes = [col for col in final_columns if col in contracts_df.columns]
contracts_df = contracts_df[colunas_finais_existentes]

snapshot_date = datetime.now().strftime("%Y-%m-%d")
contracts_df['execution_date'] = snapshot_date

# --- 4. Descaracterização e Simulação ---
FILE_KEY = f'data/acme_corp/bronze/contracts/it_contracts/{snapshot_date}.parquet'
mock_s3_manipulations("parquet_upload", contracts_df, FILE_KEY, credentials, bucket_name, None)

print("\n--- Amostra de Dados Pós-Bronze (Limpas) ---")

contracts_df[['ID', 'Analista Contrato', 'Usuario Chave', 'AtaRDG', 'ContaRazao']].head()

# --- Executando a Camada Bronze ---
# (Definimos "None" pois nossos mocks não usam credenciais reais)
bronze_layer_construction(credentials=None, bucket_name="acme-datalake")